In [ ]:
# ============================================
# BLOOD DONOR ELIGIBILITY - LOGISTIC REGRESSION (REGULARIZED + CV)
# ============================================

from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    roc_auc_score,
)
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# --------------------------------------------
# 1. Load Dataset (local first, fallback to DATASET folder)
# --------------------------------------------

candidate_paths = [
    Path("fixed_blood_donor_dataset.csv"),
    Path("../DATASET/Eligibility model/blood_donation.csv"),
]

dataset_path = next((p for p in candidate_paths if p.exists()), None)
if dataset_path is None:
    raise FileNotFoundError("Could not find fixed_blood_donor_dataset.csv")

df = pd.read_csv(dataset_path)
print(f"Loaded: {dataset_path}")
print("Dataset Shape:", df.shape)

# --------------------------------------------
# 2. Define Features and Target
# --------------------------------------------

target_col = "Eligible_for_Donation_binary"
X = df.drop(columns=[target_col])
y = df[target_col]

print("Target distribution (%):")
print((y.value_counts(normalize=True) * 100).round(2))

# --------------------------------------------
# 3. Stratified Train-Test Split
# --------------------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print("Training size:", X_train.shape)
print("Testing size:", X_test.shape)

# --------------------------------------------
# 4. Pipeline + Hyperparameter Search
# --------------------------------------------

pipeline = Pipeline(
    steps=[
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=5000, random_state=42)),
    ]
)

param_grid = {
    "model__C": [0.01, 0.1, 1.0, 5.0, 10.0],
    "model__penalty": ["l2"],
    "model__solver": ["lbfgs"],
    "model__class_weight": [None, "balanced"],
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    scoring="f1",
    cv=cv,
    n_jobs=-1,
    refit=True,
    verbose=1,
)

search.fit(X_train, y_train)
best_model = search.best_estimator_

print("\nBest Parameters:")
print(search.best_params_)
print(f"Best CV F1: {search.best_score_:.4f}")

# --------------------------------------------
# 5. Evaluate Train vs Test (Overfitting Check)
# --------------------------------------------

train_pred = best_model.predict(X_train)
test_pred = best_model.predict(X_test)
train_proba = best_model.predict_proba(X_train)[:, 1]
test_proba = best_model.predict_proba(X_test)[:, 1]

train_f1 = f1_score(y_train, train_pred)
test_f1 = f1_score(y_test, test_pred)
train_auc = roc_auc_score(y_train, train_proba)
test_auc = roc_auc_score(y_test, test_proba)

overfit_gap = train_f1 - test_f1

print(f"\nTrain Accuracy: {accuracy_score(y_train, train_pred):.4f}")
print(f"Test Accuracy:  {accuracy_score(y_test, test_pred):.4f}")
print(f"Train F1:       {train_f1:.4f}")
print(f"Test F1:        {test_f1:.4f}")
print(f"Train ROC-AUC:  {train_auc:.4f}")
print(f"Test ROC-AUC:   {test_auc:.4f}")
print(f"Overfitting Gap (Train F1 - Test F1): {overfit_gap:.4f}")

if overfit_gap > 0.05:
    print("Warning: Noticeable overfitting detected. Consider stronger regularization.")
else:
    print("Good: Overfitting gap is acceptable.")

print("\nConfusion Matrix (Test):")
print(confusion_matrix(y_test, test_pred))

print("\nClassification Report (Test):")
print(classification_report(y_test, test_pred, digits=4))

Loaded: cleaned_blood_donor_model_dataset.csv
Dataset Shape: (10000, 10)
Target distribution (%):
Eligible_for_Donation_binary
1    64.16
0    35.84
Name: proportion, dtype: float64
Training size: (8000, 9)
Testing size: (2000, 9)
Fitting 5 folds for each of 10 candidates, totalling 50 fits

Best Parameters:
{'model__C': 0.01, 'model__class_weight': None, 'model__penalty': 'l2', 'model__solver': 'lbfgs'}
Best CV F1: 1.0000

Train Accuracy: 1.0000
Test Accuracy:  1.0000
Train F1:       1.0000
Test F1:        1.0000
Train ROC-AUC:  1.0000
Test ROC-AUC:   1.0000
Overfitting Gap (Train F1 - Test F1): 0.0000
Good: Overfitting gap is acceptable.

Confusion Matrix (Test):
[[ 717    0]
 [   0 1283]]

Classification Report (Test):
              precision    recall  f1-score   support

           0     1.0000    1.0000    1.0000       717
           1     1.0000    1.0000    1.0000      1283

    accuracy                         1.0000      2000
   macro avg     1.0000    1.0000    1.0000      2

c:\Users\user\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
